In [4]:
import ast
import pandas as pd

print("Code evaluator is ready.")

Code evaluator is ready.


In [5]:
def solution(nums, target):
    seen = {}

    for index, number in enumerate(nums):
        needed = target - number

        if needed in seen:
            return [seen[needed], index]

        seen[number] = index

    return []

In [6]:
solution_code = '''
def solution(nums, target):
    seen = {}

    for index, number in enumerate(nums):
        needed = target - number

        if needed in seen:
            return [seen[needed], index]

        seen[number] = index

    return []
'''

try:
    ast.parse(solution_code)
    syntax_valid = True
    print("Syntax check: passed")
except SyntaxError as error:
    syntax_valid = False
    print("Syntax check: failed")
    print(f"Line {error.lineno}: {error.msg}")

Syntax check: passed


In [7]:
test_cases = [
    {
        "name": "Basic pair",
        "input": ([2, 7, 11, 15], 9),
        "expected": [0, 1]
    },
    {
        "name": "Pair in the middle",
        "input": ([3, 2, 4], 6),
        "expected": [1, 2]
    },
    {
        "name": "Repeated number",
        "input": ([3, 3], 6),
        "expected": [0, 1]
    },
    {
        "name": "Negative numbers",
        "input": ([-1, -2, -3, -4, -5], -8),
        "expected": [2, 4]
    },
    {
        "name": "No matching pair",
        "input": ([1, 2, 3], 10),
        "expected": []
    }
]

print("Number of test cases:", len(test_cases))

Number of test cases: 5


In [8]:
def evaluate_solution(solution_function, cases):
    results = []

    for case in cases:
        try:
            nums, target = case["input"]
            actual = solution_function(nums, target)
            passed = actual == case["expected"]

            results.append({
                "test_case": case["name"],
                "expected": case["expected"],
                "actual": actual,
                "status": "Passed" if passed else "Failed",
                "error": ""
            })

        except Exception as error:
            results.append({
                "test_case": case["name"],
                "expected": case["expected"],
                "actual": None,
                "status": "Error",
                "error": str(error)
            })

    return pd.DataFrame(results)


if syntax_valid:
    evaluation_results = evaluate_solution(solution, test_cases)
    display(evaluation_results)
else:
    print("Tests were not run because the source text has a syntax error.")

,test_case,expected,actual,status,error
0,Basic pair,"[0, 1]","[0, 1]",Passed,
1,Pair in the middle,"[1, 2]","[1, 2]",Passed,
2,Repeated number,"[0, 1]","[0, 1]",Passed,
3,Negative numbers,"[2, 4]","[2, 4]",Passed,
4,No matching pair,[],[],Passed,


In [9]:
if syntax_valid:
    passed_count = (evaluation_results["status"] == "Passed").sum()
    failed_count = (evaluation_results["status"] == "Failed").sum()
    error_count = (evaluation_results["status"] == "Error").sum()
    total_count = len(evaluation_results)
    pass_rate = (passed_count / total_count) * 100 if total_count else 0

    print("Total test cases:", total_count)
    print("Passed:", passed_count)
    print("Failed:", failed_count)
    print("Errors:", error_count)
    print(f"Pass rate: {pass_rate:.1f}%")
else:
    print("Evaluation unavailable: fix the syntax error first.")

Total test cases: 5
Passed: 5
Failed: 0
Errors: 0
Pass rate: 100.0%


In [10]:
from pathlib import Path

reports_dir = Path("../reports")
reports_dir.mkdir(parents=True, exist_ok=True)

if syntax_valid:
    report_path = reports_dir / "code_evaluation_results.csv"
    evaluation_results.to_csv(report_path, index=False)
    print("Evaluation results saved to:", report_path)
else:
    print("No report saved because syntax validation failed.")

Evaluation results saved to: ../reports/code_evaluation_results.csv


In [11]:
import os
from groq import Groq

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise EnvironmentError(
        "GROQ_API_KEY was not found. Set it in your environment before running this cell."
    )

client = Groq(api_key=api_key)


QWEN_MODEL = "qwen/qwen3.8-27b"

In [12]:
if not syntax_valid:
    raise ValueError("Fix the syntax error before requesting AI feedback.")

if "evaluation_results" not in globals():
    raise ValueError("Run the test-case evaluation cell before requesting AI feedback.")

evaluation_summary = {
    "total_test_cases": int(len(evaluation_results)),
    "passed": int((evaluation_results["status"] == "Passed").sum()),
    "failed": int((evaluation_results["status"] == "Failed").sum()),
    "errors": int((evaluation_results["status"] == "Error").sum()),
    "results": evaluation_results.to_dict(orient="records"),
}

print(evaluation_summary)

{'total_test_cases': 5, 'passed': 5, 'failed': 0, 'errors': 0, 'results': [{'test_case': 'Basic pair', 'expected': [0, 1], 'actual': [0, 1], 'status': 'Passed', 'error': ''}, {'test_case': 'Pair in the middle', 'expected': [1, 2], 'actual': [1, 2], 'status': 'Passed', 'error': ''}, {'test_case': 'Repeated number', 'expected': [0, 1], 'actual': [0, 1], 'status': 'Passed', 'error': ''}, {'test_case': 'Negative numbers', 'expected': [2, 4], 'actual': [2, 4], 'status': 'Passed', 'error': ''}, {'test_case': 'No matching pair', 'expected': [], 'actual': [], 'status': 'Passed', 'error': ''}]}


In [13]:
def get_qwen_feedback(code, test_summary, model=QWEN_MODEL):
    prompt = f"""You are a helpful Python programming tutor.

Review the Python solution and the test-case results below.

Python code:
```python
{code}
```

Test-case results (the source of truth):
{test_summary}

Please provide:
1. A short summary of what the code does.
2. What passed and what failed, based only on the supplied results.
3. Likely causes for any failures, clearly marked as possibilities when not proven.
4. Two or three practical improvement suggestions.
5. A beginner-friendly explanation.

Do not invent test results. Do not claim full correctness or performance guarantees.
If all supplied tests pass, say only that the solution passed these supplied tests.
"""

    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": "You provide accurate, concise, beginner-friendly Python code feedback."
            },
            {"role": "user", "content": prompt},
        ],
        temperature=0.2,
    )

    return response.choices[0].message.content


qwen_feedback = get_qwen_feedback(solution_code, evaluation_summary)
print(qwen_feedback)

### 1. Short Summary
The code implements the **Two Sum** algorithm. It iterates through the list `nums` once, using a dictionary (`seen`) to store numbers and their indices. For each number, it checks if the "complement" (`target - number`) has already been seen. If so, it returns the indices of the complement and the current number. If no pair is found, it returns an empty list.

### 2. Test Results
Based on the supplied results:
- **Passed:** 5 out of 5 test cases.
- **Failed:** 0 test cases.
- **Errors:** 0.

The solution passed these supplied tests.

### 3. Likely Causes for Failures
There are no failures to analyze, as all supplied tests passed.

### 4. Practical Improvement Suggestions
1. **Add Type Hints:** Adding type hints (e.g., `def solution(nums: list[int], target: int) -> list[int]:`) improves code readability and helps catch type-related bugs early.
2. **Handle Edge Cases Explicitly:** While the current code returns `[]` if no pair is found, you could add a docstring or c

In [14]:
feedback_path = reports_dir / "qwen_code_feedback.txt"
feedback_path.write_text(qwen_feedback, encoding="utf-8")

print("Qwen feedback saved to:", feedback_path)

Qwen feedback saved to: ../reports/qwen_code_feedback.txt
